# Showcasing Pathfinder for the different radar systems

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.append(os.path.abspath('../Pathfinder'))

from helper_functions import *
from radar_functions import *
from pathfinder import *
from attach_grounddata import *
from add_dataflashlog import *
from clicki_tool import *

%matplotlib inline

### Starting with the AWI WWOS GPR from 2006

In [58]:
path = "../Data/AWI/your_RADAR_object.pkl" #this was created in make_RADAR.ipynb
RADAR = load_RADAR(path=path) 

Dataset  from campaign  is loaded from ../Data/AWI/your_RADAR_object.pkl


In [59]:

# Setting the Pathfinder parameters
PF_parameters = {
    'a': 1,         #weight of radar image in cost function
    'b': 1,         #weight of (hessian) ridges in cost function
    'c': 2,         #weight of peak prominance in cost function
    'd': 2,         #weight of the localized cross-correlation 
    'db': 0,        #depth bias term (used to have a bias towards top for top layer, bottom for bottom layer)
    'sigma': 1,     #for blurring the cost map, 1 = no blurring
    'jumpiness': 1  #vertical freedom for the pathfinder (larger jumps are penalized)
    }

# Does RADAR have flightstate data?
has_flightstate = False

# Does RADAR have geolocation data?
has_geolocation = False

# Is there in-situ data (probe/ snow profile) that you want to add? Could be for validation or to view in the clicki_tool later
add_insitu_data = False
#TODO even if add_insitu_data = False, we need to add the eps_r + uncertainty! Best derived by providing a density?

# This is a UWiBaSS specific flag. Auxilliary data is added from the ardupilot dataflashlogs and some processing is done there
check_dataflashlog = False 

#TODO: Even if add_insitu_data = False, we return the 4 vars, this is of course not nice and pretty much unintuitive
RADAR, df_MP, df_SP, dict_SP = populate_datafields(RADAR,
                                     PF_parameters=PF_parameters,
                                     has_flightstate=has_flightstate,
                                     has_geolocation=has_geolocation,
                                     add_insitu_data=add_insitu_data,
                                     check_dataflashlog=check_dataflashlog,
                                     force_RPCA=False
                                     )



Computing RPCA of radar data...


In [60]:
    
first_layer = 'top' if RADAR.target_type == 'terrestrial_snow' else 'bottom' # or  RADAR.target_type == 'unknown'
second_layer = 'bottom' if first_layer == 'top' else 'top'


first_cost, first_partial_costs = compute_cost_map(RADAR.rx_rpca[:, :],
                                                   PF_parameters,
                                                   layer=first_layer,
                                                   first_layer=first_layer,
                                                   debug=True
                                                   )

second_cost, second_partial_costs = compute_next_cost_map(RADAR.rx_rpca[:, :],
                                                          PF_parameters, 
                                                          layer=second_layer,
                                                          first_layer=first_layer,
                                                          first_reflection_cost=first_partial_costs['reflection'],
                                                          first_ridge_cost=first_partial_costs['ridge'],
                                                          first_continuity_cost=first_partial_costs['continuity'],
                                                          debug=True
                                                          )

unbiased_cost = compute_next_cost_map(RADAR.rx_rpca[:, :],
                                      PF_parameters,
                                      layer=None,
                                      first_layer=None,
                                      first_reflection_cost=first_partial_costs['reflection'],
                                      first_ridge_cost=first_partial_costs['ridge'],
                                      first_continuity_cost=first_partial_costs['continuity'],
                                      debug=False
                                      )

# We can save the first cost map already.
# Since the first path will be masked in the second cost map, we save the second later.
cost_maps = {}
cost_maps[first_layer] = first_cost
cost_maps['unbiased'] = unbiased_cost

# DEPLOYING THE PATHFINDER AND DOING PATH REINEMENT
paths = {}

#find first path/layer
path, path_cost = find_optimal_path(cost_maps[first_layer], 
                                    PF_parameters,
                                    layer=first_layer,
                                    )
py_first, px_first  = list(zip(*path))
paths[first_layer] = np.array(list(py_first))


# masking the first found path in the cost map
second_cost = mask_path_in_cost(second_cost, path, radius=3, strength=.5)
cost_maps[second_layer] = second_cost

#find second path/layer
path2, path_cost2 = find_optimal_path(cost_maps[second_layer], 
                                    PF_parameters,
                                    layer=second_layer
                                    )

py_second, px_second  = list(zip(*path2))
paths[second_layer] = np.array(list(py_second))

RADAR.PF_cost_maps = cost_maps
RADAR.PF_top_interface = np.array(paths['top'])
RADAR.PF_bottom_interface = np.array(paths['bottom'])


path = '../Data/AWI'
filename = 'your_RADAR_object.pkl'

# save the RADAR object
with open(os.path.join(path, filename), 'wb') as outp:
    pickle.dump(RADAR, outp, pickle.HIGHEST_PROTOCOL)

### Next up: Data from the GEORESEARCH drone

In [61]:
path = "../Data/GEORESEARCH/your_RADAR_object.pkl"
RADAR = load_RADAR(path=path)

Dataset  from campaign  is loaded from ../Data/GEORESEARCH/your_RADAR_object.pkl


In [62]:

# Setting the Pathfinder parameters
PF_parameters = {
    'a': 1,         #weight of radar image in cost function
    'b': 1,         #weight of (hessian) ridges in cost function
    'c': 2,         #weight of peak prominance in cost function
    'd': 3,         #weight of the localized cross-correlation 
    'db': 0.1,        #depth bias term (used to have a bias towards top for top layer, bottom for bottom layer)
    'sigma': 1,     #for blurring the cost map, 1 = no blurring
    'jumpiness': 1  #vertical freedom for the pathfinder (larger jumps are penalized)
    }

# Does RADAR have flightstate data?
has_flightstate = False

# Does RADAR have geolocation data?
has_geolocation = False

# Is there in-situ data (probe/ snow profile) that you want to add? Could be for validation or to view in the clicki_tool later
add_insitu_data = False
#TODO even if add_insitu_data = False, we need to add the eps_r + uncertainty! Best derived by providing a density?

# This is a UWiBaSS specific flag. Auxilliary data is added from the ardupilot dataflashlogs and some processing is done there
check_dataflashlog = False 

#TODO: Even if add_insitu_data = False, we return the 4 vars, this is of course not nice and pretty much unintuitive
RADAR, df_MP, df_SP, dict_SP = populate_datafields(RADAR,
                                     PF_parameters=PF_parameters,
                                     has_flightstate=has_flightstate,
                                     has_geolocation=has_geolocation,
                                     add_insitu_data=add_insitu_data,
                                     check_dataflashlog=check_dataflashlog,
                                     )



In [63]:
    
first_layer = 'bottom'
second_layer = 'top'


first_cost, first_partial_costs = compute_cost_map(RADAR.rx_rpca[:, :],
                                                   PF_parameters,
                                                   layer=first_layer,
                                                   first_layer=first_layer,
                                                   debug=True
                                                   )

second_cost, second_partial_costs = compute_next_cost_map(RADAR.rx_rpca[:, :],
                                                          PF_parameters, 
                                                          layer=second_layer,
                                                          first_layer=first_layer,
                                                          first_reflection_cost=first_partial_costs['reflection'],
                                                          first_ridge_cost=first_partial_costs['ridge'],
                                                          first_continuity_cost=first_partial_costs['continuity'],
                                                          debug=True
                                                          )

unbiased_cost = compute_next_cost_map(RADAR.rx_rpca[:, :],
                                      PF_parameters,
                                      layer=None,
                                      first_layer=None,
                                      first_reflection_cost=first_partial_costs['reflection'],
                                      first_ridge_cost=first_partial_costs['ridge'],
                                      first_continuity_cost=first_partial_costs['continuity'],
                                      debug=False
                                      )

# We can save the first cost map already.
# Since the first path will be masked in the second cost map, we save the second later.
cost_maps = {}
cost_maps[first_layer] = first_cost
cost_maps['unbiased'] = unbiased_cost

# DEPLOYING THE PATHFINDER AND DOING PATH REINEMENT
paths = {}

#find first path/layer
path, path_cost = find_optimal_path(cost_maps[first_layer], 
                                    PF_parameters,
                                    layer=first_layer,
                                    )
py_first, px_first  = list(zip(*path))
paths[first_layer] = np.array(list(py_first))


# masking the first found path in the cost map
second_cost = mask_path_in_cost(second_cost, path, radius=3, strength=.5)
cost_maps[second_layer] = second_cost

#find second path/layer
path2, path_cost2 = find_optimal_path(cost_maps[second_layer], 
                                    PF_parameters,
                                    layer=second_layer
                                    )

py_second, px_second  = list(zip(*path2))
paths[second_layer] = np.array(list(py_second))

RADAR.PF_cost_maps = cost_maps
RADAR.PF_top_interface = np.array(paths['top'])
RADAR.PF_bottom_interface = np.array(paths['bottom'])


path = '../Data/GEORESEARCH'
filename = 'your_RADAR_object.pkl'

# save the RADAR object
with open(os.path.join(path, filename), 'wb') as outp:
    pickle.dump(RADAR, outp, pickle.HIGHEST_PROTOCOL)

### Last but definetly not least: The NZ Kiwi Snow Radar

In [4]:
path = "../Data/Kiwi/your_RADAR_object.pkl"
RADAR = load_RADAR(path=path)

Dataset  from campaign  is loaded from ../Data/Kiwi/your_RADAR_object.pkl


In [8]:

# Setting the Pathfinder parameters
PF_parameters = {
    'a': 1,         #weight of radar image in cost function
    'b': 1,         #weight of (hessian) ridges in cost function
    'c': 4,         #weight of peak prominance in cost function
    'd': 1,         #weight of the localized cross-correlation 
    'db': 0.1,        #depth bias term (used to have a bias towards top for top layer, bottom for bottom layer)
    'sigma': 1,     #for blurring the cost map, 1 = no blurring
    'jumpiness': 2  #vertical freedom for the pathfinder (larger jumps are penalized)
    }

# Does RADAR have flightstate data?
has_flightstate = False

# Does RADAR have geolocation data?
has_geolocation = False

# Is there in-situ data (probe/ snow profile) that you want to add? Could be for validation or to view in the clicki_tool later
add_insitu_data = False
#TODO even if add_insitu_data = False, we need to add the eps_r + uncertainty! Best derived by providing a density?

# This is a UWiBaSS specific flag. Auxilliary data is added from the ardupilot dataflashlogs and some processing is done there
check_dataflashlog = False 

#TODO: Even if add_insitu_data = False, we return the 4 vars, this is of course not nice and pretty much unintuitive
RADAR, df_MP, df_SP, dict_SP = populate_datafields(RADAR,
                                     PF_parameters=PF_parameters,
                                     has_flightstate=has_flightstate,
                                     has_geolocation=has_geolocation,
                                     add_insitu_data=add_insitu_data,
                                     check_dataflashlog=check_dataflashlog,
                                     )



In [9]:
    
first_layer = 'top'
second_layer = 'bottom'


first_cost, first_partial_costs = compute_cost_map(RADAR.rx_rpca[:, :],
                                                   PF_parameters,
                                                   layer=first_layer,
                                                   first_layer=first_layer,
                                                   debug=True
                                                   )

second_cost, second_partial_costs = compute_next_cost_map(RADAR.rx_rpca[:, :],
                                                          PF_parameters, 
                                                          layer=second_layer,
                                                          first_layer=first_layer,
                                                          first_reflection_cost=first_partial_costs['reflection'],
                                                          first_ridge_cost=first_partial_costs['ridge'],
                                                          first_continuity_cost=first_partial_costs['continuity'],
                                                          debug=True
                                                          )

unbiased_cost = compute_next_cost_map(RADAR.rx_rpca[:, :],
                                      PF_parameters,
                                      layer=None,
                                      first_layer=None,
                                      first_reflection_cost=first_partial_costs['reflection'],
                                      first_ridge_cost=first_partial_costs['ridge'],
                                      first_continuity_cost=first_partial_costs['continuity'],
                                      debug=False
                                      )

# We can save the first cost map already.
# Since the first path will be masked in the second cost map, we save the second later.
cost_maps = {}
cost_maps[first_layer] = first_cost
cost_maps['unbiased'] = unbiased_cost

# DEPLOYING THE PATHFINDER AND DOING PATH REINEMENT
paths = {}

#find first path/layer
path, path_cost = find_optimal_path(cost_maps[first_layer], 
                                    PF_parameters,
                                    layer=first_layer,
                                    )
py_first, px_first  = list(zip(*path))
paths[first_layer] = np.array(list(py_first))


# masking the first found path in the cost map
second_cost = mask_path_in_cost(second_cost, path, radius=3, strength=.5)
cost_maps[second_layer] = second_cost

#find second path/layer
path2, path_cost2 = find_optimal_path(cost_maps[second_layer], 
                                    PF_parameters,
                                    layer=second_layer
                                    )

py_second, px_second  = list(zip(*path2))
paths[second_layer] = np.array(list(py_second))

RADAR.PF_cost_maps = cost_maps
RADAR.PF_top_interface = np.array(paths['top'])
RADAR.PF_bottom_interface = np.array(paths['bottom'])


path = '../Data/Kiwi/'
filename = 'your_RADAR_object.pkl'

# save the RADAR object
with open(os.path.join(path, filename), 'wb') as outp:
    pickle.dump(RADAR, outp, pickle.HIGHEST_PROTOCOL)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
